# 02 – House prices (based on Kaggle's "Machine Learning Competitions" exercise)
Predict `SalePrice` for houses in Ames, Iowa. This is the same training data as Kaggle's Housing Prices Competition, saved in `../data/home_data.csv`.

Goal: get the **validation MAE** (mean absolute error, in dollars) as low as you can.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

home_data = pd.read_csv("../data/home_data.csv")
home_data.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


## Baseline: same features as the Kaggle exercise

In [53]:
y = home_data.SalePrice
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
X = home_data[features]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

# Decision tree
tree = DecisionTreeRegressor(max_leaf_nodes=150, random_state=1)
tree.fit(train_X, train_y)
print("Decision tree MAE:  {:,.0f}".format(mean_absolute_error(val_y, tree.predict(val_X))))

# Random forest
rf = RandomForestRegressor(random_state=1)
rf.fit(train_X, train_y)
print("Random forest MAE:  {:,.0f}".format(mean_absolute_error(val_y, rf.predict(val_X))))

Decision tree MAE:  26,988
Random forest MAE:  21,857


## Your turn: make it better
Ideas:
- Add more numeric columns to `my_features` (e.g. `OverallQual`, `GrLivArea`, `GarageCars`, `TotalBsmtSF`).
- Columns with missing values (`LotFrontage`, `GarageYrBlt`, `MasVnrArea`) need `.fillna(...)` first.
- Text columns like `Neighborhood` need `pd.get_dummies(...)` before a model can use them.
- Tune `n_estimators`, `max_depth`, `min_samples_leaf` on the forest.

In [59]:
y = home_data["SalePrice"]

all_features = home_data
X = pd.get_dummies(all_features.drop(columns=["SalePrice", "Id"]))
X = X.fillna(X.mean())
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)
my_model = RandomForestRegressor(n_estimators=600,max_depth = 20, min_samples_leaf = 1, random_state=1, max_features = .5)
my_model.fit(train_X, train_y)

print("My model MAE:  {:,.0f}".format(mean_absolute_error(val_y, my_model.predict(val_X))))

My model MAE:  15,889


In [57]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV

model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(random_state=1)
)


search = RandomizedSearchCV(
    model,
    param_distributions={
        "randomforestregressor__n_estimators": [200, 400, 600],
        "randomforestregressor__max_depth": [None, 10, 15, 20, 30],
        "randomforestregressor__min_samples_leaf": [1, 2, 4, 8],
        "randomforestregressor__max_features": [0.5, 0.75, 1.0],
    },
    n_iter=30,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=1,
    n_jobs=-1
)

search.fit(train_X, train_y)
baseline = RandomForestRegressor(
    n_estimators=201, max_depth=15, min_samples_leaf=2, random_state=1
)
baseline.fit(train_X, train_y)

print("Baseline validation MAE:",
      mean_absolute_error(val_y, baseline.predict(val_X)))
print("Search validation MAE:",
      mean_absolute_error(val_y, search.predict(val_X)))
print("Best cross-validation MAE:", -search.best_score_)
print("Validation MAE:", mean_absolute_error(val_y, search.predict(val_X)))
print("Best settings:", search.best_params_)

Baseline validation MAE: 16428.60989625318
Search validation MAE: 15888.652317151467
Best cross-validation MAE: 17558.566191826678
Validation MAE: 15888.652317151467
Best settings: {'randomforestregressor__n_estimators': 600, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__max_features': 0.5, 'randomforestregressor__max_depth': 20}


In [38]:
# Which features matter most?
pd.Series(my_model.feature_importances_, index=train_X.columns).sort_values(ascending=False)

SalePrice              0.994754
2ndFlrSF               0.000455
Exterior2nd_HdBoard    0.000258
MasVnrArea             0.000230
GarageYrBlt            0.000219
                         ...   
Utilities_AllPub       0.000000
Utilities_NoSeWa       0.000000
Street_Grvl            0.000000
PoolQC_Fa              0.000000
PoolQC_Ex              0.000000
Length: 289, dtype: float64